[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/ch6/lesson3.ipynb)

# Datos de Earth Engine

Este archivo de plantilla muestra cómo descargar datos climáticos y ambientales correspondientes a un punto, una región y un período determinados mediante Google Earth Engine.

Nota: Si todavía no has configurado Google Earth Engine, [esta lección](https://geo-di-lab.github.io/emerge-geoai-es/docs/ch1/lesson5a.html) explica cómo registrarte y obtener acceso. Google Earth Engine puede utilizarse gratuitamente para investigación, educación y actividades sin fines de lucro.

In [ ]:
import folium
import ee
import geemap
import geopandas as gpd

ee.Authenticate()

# Escribe aquí el ID de tu proyecto, entre comillas
ee.Initialize(project="emerge-lessons")


def add_ee_layer(self, ee_image_object, vis_params, name):
    """
    Agrega a un mapa de Folium una capa de teselas
    procedente de una imagen de Earth Engine.
    """
    map_id_dict = ee.Image(
        ee_image_object
    ).getMapId(vis_params)

    folium.raster_layers.TileLayer(
        tiles=map_id_dict["tile_fetcher"].url_format,
        attr=(
            'Map Data &copy; '
            '<a href="https://earthengine.google.com/">'
            "Google Earth Engine</a>"
        ),
        name=name,
        overlay=True,
        control=True
    ).add_to(self)


folium.Map.add_ee_layer = add_ee_layer

## Cargar datos de Earth Engine

Visita el [Catálogo de datos de Earth Engine](https://developers.google.com/earth-engine/datasets/catalog) y busca los conjuntos de datos que te interesen. Google Earth Engine ofrece una gran variedad de datos, como temperatura, cobertura terrestre, imágenes satelitales y agricultura.

Cuando abras la página de un conjunto de datos, busca a la derecha de la imagen la sección *Earth Engine Snippet*. El fragmento comenzará con `ee.Image` o `ee.ImageCollection`. Copia ese fragmento.

Si el fragmento comienza con `ee.ImageCollection`, el conjunto de datos contiene una [colección o secuencia de imágenes](https://developers.google.com/earth-engine/guides/ic_creating#colab-python). Por lo general, esto significa que puedes seleccionar imágenes correspondientes a distintas fechas o regiones.

Si el fragmento comienza con `ee.Image`, el conjunto contiene una sola imagen y normalmente no puede filtrarse por fecha.

Pega a continuación el fragmento de Earth Engine del conjunto de datos que deseas utilizar.

Por ejemplo, para [CFSV2: NCEP Climate Forecast System Version 2, 6-Hourly Products](https://developers.google.com/earth-engine/datasets/catalog/NOAA_CFSV2_FOR6H), el código sería:

```python
image = ee.ImageCollection("NOAA/CFSV2/FOR6H")
```

In [ ]:
# Pega aquí el fragmento de Earth Engine
image = None

# Por ejemplo:
# image = ee.ImageCollection("NOAA/CFSV2/FOR6H")

En la página del conjunto de datos de Earth Engine, desplázate hasta el final. Allí debería aparecer una sección llamada **Explore with Earth Engine** con fragmentos de código para cargar y visualizar los datos.

En algunos casos, el fragmento solamente está disponible en JavaScript. Cuando esto ocurra, copia el código de JavaScript, pégalo en la siguiente celda y ejecútala para convertirlo a Python.

In [ ]:
javascript_code = """
// Pega el código debajo de esta línea



// Pega el código encima de esta línea
"""

geemap.js_snippet_to_py(
    javascript_code,
    add_new_cell=True,
    import_ee=True,
    import_geemap=True,
    show_map=True
)

Después de ejecutar la celda anterior, aparecerá el código en Python. Busca una variable cuyo nombre contenga `vis`, como `visParams` o `maximumTemperatureVis`.

Si encuentras esa variable, copia todo el contenido ubicado entre las llaves `{}` y pégalo dentro de las llaves de `visual_params = {}` en la siguiente celda. Utilizaremos estos parámetros más adelante para visualizar los datos en un mapa.

Si no existe una variable de visualización, deja `visual_params = {}`.

Antes de ejecutar el código convertido, busca cualquier uso de `ee.Filter.date()` u otras fechas. Puedes reemplazar esas fechas por el período que deseas estudiar. Al ejecutar el código, debería aparecer un mapa con los datos.

In [ ]:
visual_params = {}

En el catálogo de Earth Engine, abre la sección **Bands** del conjunto de datos. Al principio de esa sección aparecerá el **Pixel Size**. Escribe ese tamaño de píxel, en metros, en la siguiente celda.

In [ ]:
# Escribe el tamaño del píxel en metros
pixel_size = None

# Por ejemplo:
# pixel_size = 22264

En la sección **Bands**, revisa la lista de bandas y elige la que más te interese. Escribe el nombre exacto de la banda a continuación.

Por ejemplo, el conjunto de datos de pronóstico climático contiene bandas como `Temperature_height_above_ground` y `Surface_pressure`.

In [ ]:
band_name = ""

## Obtener datos para un punto específico

Si tu conjunto de datos es un `ee.ImageCollection`, escribe las fechas inicial y final con el formato `aaaa-mm-dd`, como `2025-06-01`.

La fecha inicial está incluida y la fecha final está excluida. Por ejemplo, el siguiente código selecciona imágenes desde el 1 de junio de 2025 hasta el 30 de junio de 2025:

```python
start_date = "2025-06-01"
end_date = "2025-07-01"
```

Si tu conjunto de datos es un `ee.Image`, puedes dejar ambas fechas vacías.

In [ ]:
start_date = ""
end_date = ""

Escribe la latitud y la longitud del lugar que deseas analizar.

Una manera de encontrar estas coordenadas es abrir [Google Maps](https://www.google.com/maps), hacer clic con el botón izquierdo sobre el lugar y copiar las coordenadas que aparecen.

In [ ]:
latitude = None
longitude = None

# Por ejemplo:
# latitude = 25.93805
# longitude = -80.78048

En este ejemplo utilizaremos los siguientes valores. Puedes usar los mismos o reemplazarlos por otros.

```python
image = ee.ImageCollection("NOAA/CFSV2/FOR6H")

visual_params = {
    "min": 220.0,
    "max": 310.0,
    "palette": ["blue", "purple", "cyan", "green", "yellow", "red"],
}

pixel_size = 22264
band_name = "Temperature_height_above_ground"

start_date = "2025-06-01"
end_date = "2025-07-01"

latitude = 25.93805
longitude = -80.78048
```

Comprueba que las coordenadas sean correctas. Si el punto aparece en un lugar inesperado o sobre el agua, es posible que hayas intercambiado la latitud y la longitud.

En Florida, la latitud generalmente se encuentra entre 24 y 31, mientras que la longitud suele estar entre -80 y -88.

In [ ]:
if latitude is None or longitude is None:
    raise ValueError(
        "Escribe valores numéricos para latitude y longitude."
    )

if not -90 <= latitude <= 90:
    raise ValueError(
        "La latitud debe encontrarse entre -90 y 90."
    )

if not -180 <= longitude <= 180:
    raise ValueError(
        "La longitud debe encontrarse entre -180 y 180."
    )

point = ee.Geometry.Point(
    longitude,
    latitude
)

map = folium.Map(
    location=[latitude, longitude],
    tiles="OpenStreetMap",
    zoom_start=10
)

folium.Marker(
    location=[latitude, longitude]
).add_to(map)

display(map)

Utilizaremos la información seleccionada en la primera parte del cuaderno para extraer los datos correspondientes al período y las coordenadas indicados.

En este ejemplo se utiliza `.median()` para resumir una colección de imágenes. Según el conjunto de datos, también puedes utilizar `.mean()`, `.sum()`, `.min()` o `.max()`.

In [ ]:
if image is None:
    raise ValueError(
        "Primero asigna un ee.Image o ee.ImageCollection a image."
    )
if pixel_size is None:
    raise ValueError(
        "Escribe el tamaño del píxel en pixel_size."
    )
if not band_name:
    raise ValueError(
        "Escribe el nombre de una banda en band_name."
    )


def prepare_image(
    image_object,
    start_date="",
    end_date=""
):
    """
    Devuelve una sola imagen lista para el análisis.

    Si el objeto es una colección, filtra por fecha y
    calcula la mediana. Si ya es una imagen, la devuelve
    sin aplicar un filtro temporal.
    """
    if isinstance(
        image_object,
        ee.imagecollection.ImageCollection
    ):
        if not start_date or not end_date:
            raise ValueError(
                "Una colección de imágenes requiere "
                "start_date y end_date."
            )

        return (
            image_object
            .filterDate(start_date, end_date)
            .median()
        )

    if isinstance(
        image_object,
        ee.image.Image
    ):
        return image_object

    raise TypeError(
        "image debe ser un ee.Image o ee.ImageCollection."
    )


selected_image = prepare_image(
    image,
    start_date,
    end_date
)

data_at_point = (
    selected_image
    .sample(
        point,
        pixel_size
    )
    .first()
    .get(band_name)
    .getInfo()
)

if data_at_point is None:
    raise ValueError(
        "No se encontró un valor para el punto, la banda "
        "y el período seleccionados."
    )

date_description = (
    f"del {start_date} al {end_date}"
    if start_date and end_date
    else "en la imagen seleccionada"
)

print(
    f"Para el período {date_description}, el valor de la banda "
    f'"{band_name}" en el punto seleccionado es {data_at_point}.'
)

## Obtener datos para una región

Para obtener datos de una región, primero debes cargar sus límites geográficos. Por ejemplo, puedes utilizar los límites de un estado, un condado o una ciudad.

En este ejemplo utilizaremos los límites del estado de Florida.

Cargaremos un archivo con los límites de los condados de Florida basado en datos descargados previamente de la [Oficina del Censo de Estados Unidos](https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html).

Después, uniremos los condados para formar una sola región estatal.

In [ ]:
fl = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/raw/"
    "refs/heads/main/docs/data/florida_counties.geojson"
)

fl_state = fl.dissolve()

Convierte los límites de Florida al formato de Earth Engine.

In [ ]:
region_features = geemap.geopandas_to_ee(
    fl_state
)

# Obtener una sola geometría para recortar imágenes
# y calcular estadísticas regionales
region = region_features.geometry()

Obtén los datos correspondientes a toda la región.

In [ ]:
if image is None:
    raise ValueError(
        "Primero asigna un ee.Image o ee.ImageCollection a image."
    )
if not band_name:
    raise ValueError(
        "Escribe el nombre de una banda en band_name."
    )

selected_image = prepare_image(
    image,
    start_date,
    end_date
)

data_at_region = (
    selected_image
    .select(band_name)
    .clip(region)
)

Muestra los datos de la región en un mapa.

In [ ]:
map = folium.Map(
    location=[28.263363, -83.497652],
    tiles="OpenStreetMap",
    zoom_start=7
)

map.add_ee_layer(
    data_at_region,
    visual_params,
    band_name or "Datos de Earth Engine"
)

folium.LayerControl(
    collapsed=False
).add_to(map)

display(map)

Calcula el valor promedio de la banda seleccionada en toda la región.

In [ ]:
if pixel_size is None:
    raise ValueError(
        "Escribe el tamaño del píxel en pixel_size."
    )

average_over_region = (
    data_at_region
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=pixel_size,
        maxPixels=1e12
    )
    .get(band_name)
    .getInfo()
)

if average_over_region is None:
    raise ValueError(
        "No se encontró un promedio para la región, la banda "
        "y el período seleccionados."
    )

date_description = (
    f"del {start_date} al {end_date}"
    if start_date and end_date
    else "en la imagen seleccionada"
)

print(
    f"El promedio de la banda {band_name} para la región "
    f"{date_description} es {average_over_region}."
)

Este cuaderno muestra cómo cargar datos de Google Earth Engine y extraer información para un punto y una región. Consulta estos recursos para obtener más información:

- [Tutoriales del paquete geemap](https://geemap.org/tutorials/#geemap-tutorials)
- [Introducción a la API de Python de Earth Engine](https://developers.google.com/earth-engine/tutorials/community/intro-to-python-api)